# UCS420: Cognitive Computing
## Assignment 8 — NLP using Python-II

**Name:** Parth Khurana  
**Roll Number:** 1024170354  
**Batch:** 3Q33


## Setup

In [ ]:
import re, string
from collections import Counter
import numpy as np, pandas as pd, nltk, matplotlib.pyplot as plt
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
for resource in ["punkt","punkt_tab","stopwords","wordnet","omw-1.4"]:
    nltk.download(resource,quiet=True)
stop_words=set(stopwords.words("english"))

## Q1. Paragraph preprocessing

In [ ]:
paragraph = ("Artificial intelligence is my favorite technology because it helps people solve complex problems. "
"Machine learning systems discover patterns from data and improve predictions over time. "
"I enjoy learning how algorithms turn raw information into useful decisions. "
"Responsible AI should be transparent, fair, and secure. "
"In the future, intelligent tools may improve education, healthcare, and everyday productivity.")
print("Original:",paragraph)
clean=re.sub(r"[^a-z0-9\s]","",paragraph.lower())
print("Lowercase, punctuation removed:",clean)
print("split():",clean.split())
print("word_tokenize():",word_tokenize(clean))
print("Sentences:",sent_tokenize(paragraph))
tokens=word_tokenize(clean)
filtered=[w for w in tokens if w.isalpha() and w not in stop_words]
print("After stopword removal:",filtered)
print("Frequency distribution:",Counter(filtered))
print("Comparison: split() separates on whitespace; word_tokenize() applies tokenizer rules. Since punctuation was removed first, outputs may be similar.")

## Q2. Regex, stemming and lemmatization

In [ ]:
words=re.findall(r"\b[A-Za-z]+\b",paragraph.lower())
without_stop=[w for w in words if w not in stop_words]
porter=PorterStemmer(); lemma=WordNetLemmatizer()
comparison=pd.DataFrame({"word":without_stop,"stemmed":[porter.stem(w) for w in without_stop],
                         "lemmatized":[lemma.lemmatize(w) for w in without_stop]})
display(comparison)
print("Stemming applies rules to remove endings and may create non-words. Lemmatization aims to return a dictionary base form. Stemming is fast for approximate search; lemmatization is preferable when linguistic readability matters.")

## Q3. Bag of Words and TF-IDF

In [ ]:
texts=[
"The laptop battery lasts long and the screen is bright.",
"The phone camera takes sharp photos and the battery charges quickly.",
"The laptop is fast, lightweight, and useful for programming."
]
cv=CountVectorizer(); bow=cv.fit_transform(texts)
display(pd.DataFrame(bow.toarray(),columns=cv.get_feature_names_out(),index=["Text 1","Text 2","Text 3"]))
tv=TfidfVectorizer(stop_words="english"); mat=tv.fit_transform(texts)
terms=np.array(tv.get_feature_names_out()); values=mat.toarray()
display(pd.DataFrame(values,columns=terms,index=["Text 1","Text 2","Text 3"]).round(3))
for i,row in enumerate(values):
    ids=row.argsort()[::-1][:3]
    print(f"Top 3 keywords Text {i+1}:",[(terms[j],round(float(row[j]),3)) for j in ids])

## Q4. Jaccard and cosine similarity

In [ ]:
tech1="Artificial intelligence learns patterns from data. AI systems classify images, understand language, and make predictions. They are used in education and healthcare."
tech2="Blockchain stores records in linked blocks. A distributed ledger can improve transparency and make transactions difficult to alter. Blockchain is used in digital payments and supply chains."
def prep(s): return [w for w in re.findall(r"\b[a-z]+\b",s.lower()) if w not in stop_words]
a,b=set(prep(tech1)),set(prep(tech2))
jaccard=len(a&b)/len(a|b) if a|b else 0
tf=TfidfVectorizer(stop_words="english"); mx=tf.fit_transform([tech1,tech2])
cos=float(cosine_similarity(mx[0:1],mx[1:2])[0,0])
print("Jaccard similarity:",jaccard)
print("Cosine similarity:",cos)
print("Analysis: Jaccard compares unique terms and ignores frequency. Cosine compares TF-IDF-weighted vectors, often giving a more nuanced comparison of informative shared terms.")

## Q5. Review sentiment and word cloud

In [ ]:
reviews=[
"This laptop is excellent, fast, and easy to use. I love the bright display.",
"The service was wonderful and the staff were helpful.",
"The battery is terrible and the device gets hot. Very disappointing.",
"The delivery was slow and the product arrived damaged.",
"The product is okay and works as expected."
]
rows=[]
try:
    from textblob import TextBlob
    method="TextBlob"
    for r in reviews:
        s=TextBlob(r).sentiment; label="Positive" if s.polarity>0.05 else "Negative" if s.polarity< -0.05 else "Neutral"
        rows.append({"review":r,"polarity":s.polarity,"subjectivity":s.subjectivity,"sentiment":label})
except Exception:
    import nltk
    nltk.download("vader_lexicon",quiet=True)
    from nltk.sentiment import SentimentIntensityAnalyzer
    analyzer=SentimentIntensityAnalyzer(); method="VADER"
    for r in reviews:
        score=analyzer.polarity_scores(r)["compound"]
        label="Positive" if score>=0.05 else "Negative" if score<=-0.05 else "Neutral"
        rows.append({"review":r,"polarity/compound":score,"subjectivity":np.nan,"sentiment":label})
df=pd.DataFrame(rows); print("Method:",method); display(df)
positive=" ".join(df.loc[df.sentiment=="Positive","review"])
try:
    from wordcloud import WordCloud
    if positive.strip():
        plt.figure(figsize=(12,6)); plt.imshow(WordCloud(width=900,height=450,background_color="white").generate(positive),interpolation="bilinear")
        plt.axis("off"); plt.title("Word cloud — positive reviews"); plt.show()
    else: print("No positive reviews found.")
except ImportError:
    print("Install wordcloud with: pip install wordcloud")

## Q6. Text generation with Keras

In [ ]:
training_text=("Artificial intelligence helps computers learn from examples. A neural network finds patterns in data and uses those patterns to make predictions. "
"Language models learn sequences of words from text. During training, the model studies which words often appear together. "
"A small model cannot understand the world, but it can learn simple patterns. With more examples and careful evaluation, text generation can become more coherent. "
"Responsible systems should be tested for quality, safety, and bias.")
try:
    from tensorflow.keras.preprocessing.text import Tokenizer
    from tensorflow.keras.preprocessing.sequence import pad_sequences
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Embedding, LSTM, Dense
    from tensorflow.keras.utils import to_categorical
    tokenizer=Tokenizer(); tokenizer.fit_on_texts([training_text])
    seq=tokenizer.texts_to_sequences([training_text])[0]
    index_word={i:w for w,i in tokenizer.word_index.items()}
    examples=[seq[:i+1] for i in range(1,len(seq))]
    maxlen=max(len(x) for x in examples)
    padded=pad_sequences(examples,maxlen=maxlen,padding="pre")
    Xtrain=padded[:,:-1]; ytrain=to_categorical(padded[:,-1],num_classes=len(tokenizer.word_index)+1)
    model=Sequential([Embedding(len(tokenizer.word_index)+1,16),LSTM(32),Dense(len(tokenizer.word_index)+1,activation="softmax")])
    model.compile(optimizer="adam",loss="categorical_crossentropy",metrics=["accuracy"])
    model.fit(Xtrain,ytrain,epochs=100,verbose=0)
    def generate(seed,n=10):
        result=seed.lower()
        for _ in range(n):
            ids=tokenizer.texts_to_sequences([result])[0]
            padded_seed=pad_sequences([ids],maxlen=maxlen-1,padding="pre")
            next_id=int(np.argmax(model.predict(padded_seed,verbose=0)[0]))
            word=index_word.get(next_id,"")
            if not word: break
            result += " "+word
        return result
    print(generate("artificial intelligence",10))
    print(generate("a neural network",10))
    print(generate("language models",10))
except Exception as e:
    print("TensorFlow is required. In Colab run: !pip install tensorflow")
    print("Then rerun this cell. Error:",repr(e))

**Q6 note:** Because the training paragraph is short, generated text may repeat words or be grammatically weak. This is expected for a small demonstration model.